# ♟️ Chess Numba — Google Colab 雲端搜尋調參執行器

本筆記本專為 **Google Colab VS Code Extension** 設計：
- 在本地 VS Code 編輯代碼
- 運算核心切換為 **Google Colab 遠端運算**（本機 CPU 使用率 0%）
- 調參目標：**方案 B（LMR 走法獎懲與晚期剪枝全套六軸）**

### 💡 在 VS Code 中連接 Colab 核心步驟：
1. 點擊本編輯器右上角的 **「Select Kernel（選取核心）」**
2. 選擇 **「Colab」**
3. 登入 Google 帳號授權，選取一個運行階段（Runtime）
4. 依序執行下方 Cell 即可開始雲端自動化調參！

In [ ]:
# 1. 檢視 Colab 雲端硬體規格
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!python --version

In [ ]:
# 2. 克隆/拉取最新代碼庫並安裝相依套件
import os
branch = "feat-initial-bitboard-setup"

if not os.path.exists("/content/chess_numba"):
    %cd /content
    !git clone -b {branch} https://github.com/b08203031/chess_numba.git

%cd /content/chess_numba
!git checkout {branch}
!git pull origin {branch}
!pip install -q numba chess numpy

In [ ]:
# 3. 測試環境與常數載入
import sys
sys.path.insert(0, "/content/chess_numba")
from tune_search.search_param_registry import get_default_params

target_params = [
    "LMR_BAD_CAPTURE_BONUS",
    "LMR_GOOD_CAPTURE_RELIEF",
    "LMR_KILLER_COUNTER_RELIEF",
    "LMR_NO_TTMOVE_BONUS",
    "LMP_SCALE_PERCENT",
    "LMR_NOT_IMP_NUM"
]
print("✅ 成功載入！方案 B 全套六軸基準預設值：")
print(get_default_params(names=target_params))

In [ ]:
# 4. 啟動 L1 SPSA 方案 B 雲端對抗調參（30 輪 x 60 局 = 1,800 局）
import os
cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"偵測到 Colab CPU 核心數: {cpu_count}，調參並行度設為: {concurrency}")

!python -m tune_search.search_tuner \
  --params LMR_BAD_CAPTURE_BONUS,LMR_GOOD_CAPTURE_RELIEF,LMR_KILLER_COUNTER_RELIEF,LMR_NO_TTMOVE_BONUS,LMP_SCALE_PERCENT,LMR_NOT_IMP_NUM \
  --backend inprocess \
  --scale-mode integer \
  --iter 30 \
  --games 60 \
  --nodes 50000 \
  --concurrency {concurrency} \
  --adjudicate \
  --log tune_search/logs/lmr_p2_l1.jsonl \
  --state tune_search/states/lmr_p2_l1.json

In [ ]:
# 5. 分析 SPSA 調參成果，以 Polyak-Ruppert 後半段平均產出最佳候選參數
!python tune_search/summarize_run.py tune_search/logs/lmr_p2_l1.jsonl --tail 0.5 --out tune_search/candidate_lmr_p2.json
print("\n=== 候選參數 JSON 內容 ===")
!cat tune_search/candidate_lmr_p2.json

In [ ]:
# 6. 直接在雲端進行 L2 深度 SPRT 實戰驗證（200,000 nodes, 300 局）
!python tools/run_search_param_match.py \
  --params-json tune_search/candidate_lmr_p2.json \
  --nodes 200000 \
  --games 300 \
  --concurrency {concurrency} \
  --sprt 0,3 \
  --batch-games 50 \
  --adjudicate \
  --out tournament_analysis/l2_verify_lmr_p2.json